# Pipeline: Landing → Bronze

Este notebook implementa el pipeline de ingestión desde la **landing zone** hacia la capa **Bronze**.

### ¿Qué hace este pipeline?
1. **Extrae** los archivos CSV desde el volume `/Volumes/landing/origin_system/raw_data/`
2. **Valida** las claves naturales (unicidad + no nulidad) de cada tabla
3. **Agrega columnas de sistema** (`PK_HASH`, `R_HASH`, `CREATED_AT`, `UPDATED_AT`, `DELETED`, `DELETED_AT`)
4. **Carga** cada tabla en Bronze mediante **UPSERT** (Merge en Delta Lake)

### Tablas generadas
| Tabla Bronze                        | Archivo Fuente      | PK              |
|-------------------------------------|---------------------|-----------------|
| `bronze.origin_system.venta`        | `Venta.csv`         | `IdVenta`       |
| `bronze.origin_system.clientes`     | `Clientes.csv`      | `ID`            |
| `bronze.origin_system.productos`    | `Productos.csv`     | `ID_PRODUCTO`   |
| `bronze.origin_system.sucursales`   | `Sucursales.csv`    | `ID`            |
| `bronze.origin_system.empleados`    | `Empleados.csv`     | `ID_EMPLEADO`   |
| `bronze.origin_system.compra`       | `Compra.csv`        | `IdCompra`      |
| `bronze.origin_system.gasto`        | `Gasto.csv`         | `IdGasto`       |
| `bronze.origin_system.tipos_gasto`  | `TiposDeGasto.csv`  | `IdTipoGasto`   |
| `bronze.origin_system.canal_venta`  | `CanalDeVenta.csv`  | `IdCanal`       |
| `bronze.origin_system.proveedores`  | `Proveedores.csv`   | `IdProveedor`   |
| `bronze.origin_system.calendario`   | `Calendario.csv`    | `Fecha`         |

> ⚠️ **Prerequisito**: Ejecutar los scripts en `configs/` antes de correr este pipeline.

### Imports de librerías

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.dataframe import DataFrame
from pyspark.errors import AnalysisException
from delta.tables import *
from typing import Dict, List
import logging

### Configuración de la sesión Spark

In [0]:
spark = (
    SparkSession.builder
    .appName('LandingToBronzeApp')
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

logging.basicConfig(level=logging.INFO)

### Parámetros del pipeline

In [0]:
# Catálogo fuente (landing zone — volumes)
LANDING_CATALOG = "landing"
LANDING_SCHEMA  = "origin_system"
LANDING_VOLUME  = "raw_data"
LANDING_PATH    = f"/Volumes/{LANDING_CATALOG}/{LANDING_SCHEMA}/{LANDING_VOLUME}"

# Catálogo destino (bronze — Delta tables)
BRONZE_CATALOG = "bronze"
BRONZE_SCHEMA  = "origin_system"

print(f"Landing path : {LANDING_PATH}")
print(f"Bronze target: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")

---
## Carga de funciones utilitarias

Las funciones reutilizables están centralizadas en `utils/functions/`. Usamos `%run` para
ejecutarlas en el contexto actual del notebook, haciendo sus definiciones disponibles aquí.

| Módulo | Función disponible | Descripción |
|---|---|---|
| `validate_natural_key` | `validate_natural_key_df` | Valida unicidad y no nulidad de PKs |
| `prepare_system_columns` | `prepare_system_dataframe` | Agrega columnas de sistema (hashes, timestamps) |
| `upsert` | `upsert_data` | Realiza INSERT o MERGE según existencia de tabla |

In [0]:
%run ../utils/functions/validate_natural_key

In [0]:
%run ../utils/functions/prepare_system_columns

In [0]:
%run ../utils/functions/upsert

---
## Paso 1 — Extracción desde Landing Zone

Leemos los archivos CSV desde el volume de landing. Cada archivo tiene su configuración
de separador específica según el sistema de origen.

In [0]:
# Opciones de lectura comunes para archivos con separador coma
csv_opts_comma = {"header": "true", "inferSchema": "true", "sep": ",", "encoding": "UTF-8"}

# Opciones de lectura para archivos con separador punto y coma (sistema legado)
csv_opts_semicolon = {"header": "true", "inferSchema": "true", "sep": ";", "encoding": "ISO-8859-1"}

def read_csv(filename: str, opts: dict) -> DataFrame:
    """Lee un CSV desde la landing zone con las opciones dadas."""
    path = f"{LANDING_PATH}/{filename}"
    return spark.read.format("csv").options(**opts).load(path)

# --- Lectura de todos los archivos fuente ---
df_venta        = read_csv("Venta.csv",        csv_opts_comma)
df_clientes     = read_csv("Clientes.csv",     csv_opts_semicolon)
df_productos    = read_csv("Productos.csv",    csv_opts_comma)
df_sucursales   = read_csv("Sucursales.csv",   csv_opts_semicolon)
df_empleados    = read_csv("Empleados.csv",    csv_opts_comma)
df_compra       = read_csv("Compra.csv",       csv_opts_comma)
df_gasto        = read_csv("Gasto.csv",        csv_opts_comma)
df_tipos_gasto  = read_csv("TiposDeGasto.csv", csv_opts_comma)
df_canal_venta  = read_csv("CanalDeVenta.csv", csv_opts_comma)
df_proveedores  = read_csv("Proveedores.csv",  csv_opts_comma)
df_calendario   = read_csv("Calendario.csv",   csv_opts_comma)

print("✅ Todos los archivos CSV leídos desde la landing zone.")

---
## Paso 2 — Verificación de schemas

Inspeccionamos los esquemas de los DataFrames para asegurarnos de que la inferencia
de tipos fue correcta antes de avanzar.

In [0]:
for nombre, df in [
    ("Venta",       df_venta),
    ("Clientes",    df_clientes),
    ("Productos",   df_productos),
    ("Sucursales",  df_sucursales),
    ("Empleados",   df_empleados),
    ("Compra",      df_compra),
    ("Gasto",       df_gasto),
    ("TiposDeGasto",df_tipos_gasto),
    ("CanalDeVenta",df_canal_venta),
    ("Proveedores", df_proveedores),
    ("Calendario",  df_calendario),
]:
    print(f"{'=' * 50}\nSCHEMA: {nombre}")
    df.printSchema()

---
## Paso 3 — Validación de claves naturales

Usamos `validate_natural_key_df` de `utils/functions/validate_natural_key.py`.

Verifica que cada tabla tenga una **clave primaria válida**: sin nulos, sin duplicados.

> Si esta celda lanza un `ValueError`, hay un problema de calidad de datos en el archivo fuente.

In [0]:
validate_natural_key_df({
    "venta"       : (df_venta,       ["IdVenta"]),
    "clientes"    : (df_clientes,    ["ID"]),
    "productos"   : (df_productos,   ["ID_PRODUCTO"]),
    "sucursales"  : (df_sucursales,  ["ID"]),
    "empleados"   : (df_empleados,   ["ID_empleado"]),
    "compra"      : (df_compra,      ["IdCompra"]),
    "gasto"       : (df_gasto,       ["IdGasto"]),
    "tipos_gasto" : (df_tipos_gasto, ["IdTipoGasto"]),
    "canal_venta" : (df_canal_venta, ["IdCanal"]),
    "proveedores" : (df_proveedores, ["IdProveedor"]),
    "calendario"  : (df_calendario,  ["Fecha"]),
})

print("\n✅ Todas las claves naturales son válidas. Avanzando a Bronze.")

---
## Paso 4 — Preparación de DataFrames con columnas de sistema

Usamos `prepare_system_dataframe` de `utils/functions/prepare_system_columns.py`.

Agrega a cada DataFrame: `PK_HASH`, `R_HASH`, `CREATED_AT`, `UPDATED_AT`, `DELETED`, `DELETED_AT`.

In [0]:
bronze_venta       = prepare_system_dataframe(df_venta,       ["IdVenta"])
bronze_clientes    = prepare_system_dataframe(df_clientes,    ["ID"])
bronze_productos   = prepare_system_dataframe(df_productos,   ["ID_PRODUCTO"])
bronze_sucursales  = prepare_system_dataframe(df_sucursales,  ["ID"])
bronze_empleados   = prepare_system_dataframe(df_empleados,   ["ID_empleado"])
bronze_compra      = prepare_system_dataframe(df_compra,      ["IdCompra"])
bronze_gasto       = prepare_system_dataframe(df_gasto,       ["IdGasto"])
bronze_tipos_gasto = prepare_system_dataframe(df_tipos_gasto, ["IdTipoGasto"])
bronze_canal_venta = prepare_system_dataframe(df_canal_venta, ["IdCanal"])
bronze_proveedores = prepare_system_dataframe(df_proveedores, ["IdProveedor"])
bronze_calendario  = prepare_system_dataframe(df_calendario,  ["Fecha"])

print("✅ DataFrames preparados con columnas de sistema.")
print("\nEjemplo — schema de bronze_venta:")
bronze_venta.printSchema()

---
## Paso 5 — Carga a Bronze (UPSERT)

Usamos `upsert_data` de `utils/functions/upsert.py`.

El orden de carga respeta primero las tablas de maestros (dimensiones) y luego las de hechos (transacciones).

#### 5a. Tablas de maestros / dimensiones

In [0]:
upsert_data(bronze_clientes,    f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.clientes")
upsert_data(bronze_productos,   f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.productos")
upsert_data(bronze_sucursales,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.sucursales")
upsert_data(bronze_empleados,   f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.empleados")
upsert_data(bronze_canal_venta, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.canal_venta")
upsert_data(bronze_proveedores, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.proveedores")
upsert_data(bronze_tipos_gasto, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.tipos_gasto")
upsert_data(bronze_calendario,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.calendario")

print("\n✅ Tablas de maestros cargadas en Bronze.")

#### 5b. Tablas de hechos / transacciones

In [0]:
upsert_data(bronze_venta,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.venta")
upsert_data(bronze_compra, f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.compra")
upsert_data(bronze_gasto,  f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.gasto")

print("\n✅ Tablas de hechos cargadas en Bronze.")

---
## Paso 6 — Verificación de carga

In [0]:
print(f"Tablas en {BRONZE_CATALOG}.{BRONZE_SCHEMA}:")
spark.sql(f"SHOW TABLES IN {BRONZE_CATALOG}.{BRONZE_SCHEMA}").show()

tablas = [
    "venta", "clientes", "productos", "sucursales", "empleados",
    "compra", "gasto", "tipos_gasto", "canal_venta", "proveedores", "calendario"
]

print("\nConteo de registros por tabla:")
print("-" * 45)
for tabla in tablas:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{tabla}").count()
    print(f"  {tabla:<20} : {count:>10,} registros")
print("-" * 45)

---
## Pipeline completado ✅

Los datos de la landing zone fueron procesados y cargados en Bronze.

**Próximo paso**: ejecutar el pipeline `sales_pipeline_silver` que toma los datos de
`bronze.origin_system` y genera la tabla enriquecida `silver.sales.total_sales`.